# Load model

In [ ]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install --upgrade --no-cache-dir transformers

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.6.1: Fast Llama patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.6.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Load dataset

In [ ]:
import json

with open('/content/drive/MyDrive/stigma_project/data/healthunlocked-data/sft_sample.json', 'r') as f:
  data = json.loads(f.read())
  f.close()

In [ ]:
data[0]

{'post': 'I have recently finished my 12 month treatment for V& O. \nI am pre diabetic and just had my yearly checkup. My blood sugars are now in the normal range.i.e. not diabetic. I also find that my blood pressure which was normally borderline high has during my treatment gone to a healthy reading. Is this a temporary abberation caused by the treatment. Will my usual problems re-appear once the effects of the V&O clear my system. I am interested to know.\n',
 'label': 'neutral'}

In [ ]:
dataset = []
for d in data:
  chat = {'conversations': []}
  chat['conversations'].append({'role': 'system',
               'content': "You are an advanced text analysis AI designed to identify and label social media content for any signs of stigmatized language or messaging regarding people with diabetes. You have expertise in understanding bias, stereotypes, and stigmatizing terms, and you apply your knowledge impartially. When presented with a social media post, you will classify the content based on whether it contains: 1. **Stigmatized Content**: Language that is discriminatory, biased, shaming, or marginalizing towards individuals with diabetes. 2. **Neutral Content**: Language that is neutral, factual, or supportive without stigmatizing. For each post, you must provide: 1. **Label**: 'Stigmatized Content' or 'Neutral Content.' 2. **Explanation**: A brief rationale for why the post was labeled this way. Your decisions must be based on the specific words and tone of the post, avoiding personal interpretation or assumptions about the author's intent unless explicitly stated in the text."})

  chat['conversations'].append({'role': 'user',
               'content': f"Here is a social media post. Please classify it as either 'Stigmatized Content' or 'Neutral Content' based on its language and tone regarding people with diabetes. **Post main body**: {d['post']}"})

  chat['conversations'].append({'role': 'assistant',
               'content': f"This post includes {d['label']} content."})
  dataset.append(chat)

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(dataset)
print(dataset)

Dataset({
    features: ['conversations'],
    num_rows: 5000
})


In [ ]:
dataset[0]['conversations']

[{'content': "You are an advanced text analysis AI designed to identify and label social media content for any signs of stigmatized language or messaging regarding people with diabetes. You have expertise in understanding bias, stereotypes, and stigmatizing terms, and you apply your knowledge impartially. When presented with a social media post, you will classify the content based on whether it contains: 1. **Stigmatized Content**: Language that is discriminatory, biased, shaming, or marginalizing towards individuals with diabetes. 2. **Neutral Content**: Language that is neutral, factual, or supportive without stigmatizing. For each post, you must provide: 1. **Label**: 'Stigmatized Content' or 'Neutral Content.' 2. **Explanation**: A brief rationale for why the post was labeled this way. Your decisions must be based on the specific words and tone of the post, avoiding personal interpretation or assumptions about the author's intent unless explicitly stated in the text.",
  'role': 's

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

In [ ]:
from unsloth.chat_templates import standardize_sharegpt
# ds1 = standardize_sharegpt(dataset)
dataset2 = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

# Train the model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset2,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(), # not is_bfloat16_supported()
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

"<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are an advanced text analysis AI designed to identify and label social media content for any signs of stigmatized language or messaging regarding people with diabetes. You have expertise in understanding bias, stereotypes, and stigmatizing terms, and you apply your knowledge impartially. When presented with a social media post, you will classify the content based on whether it contains: 1. **Stigmatized Content**: Language that is discriminatory, biased, shaming, or marginalizing towards individuals with diabetes. 2. **Neutral Content**: Language that is neutral, factual, or supportive without stigmatizing. For each post, you must provide: 1. **Label**: 'Stigmatized Content' or 'Neutral Content.' 2. **Explanation**: A brief rationale for why the post was labeled this way. Your decisions must be based on the specific words and tone of th

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

KeyError: 'labels'

In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
3.441 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856/3,000,000,000 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.734400
2,3.089300
3,2.847400
4,2.827300
5,2.743100
6,2.777300
7,2.436700
8,2.403700
9,2.157200
10,2.274200


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

387.1956 seconds used for training.
6.45 minutes used for training.
Peak reserved memory = 4.051 GB.
Peak reserved memory for training = 0.61 GB.
Peak reserved memory % of max memory = 27.481 %.
Peak reserved memory for training % of max memory = 4.138 %.


# Inference

In [ ]:
messages = [
    {'role': 'system',
               'content': "You are an advanced text analysis AI designed to identify and label social media content for any signs of stigmatized language or messaging regarding people with diabetes. You have expertise in understanding bias, stereotypes, and stigmatizing terms, and you apply your knowledge impartially. When presented with a social media post, you will classify the content based on whether it contains: 1. **Stigmatized Content**: Language that is discriminatory, biased, shaming, or marginalizing towards individuals with diabetes. 2. **Neutral Content**: Language that is neutral, factual, or supportive without stigmatizing. For each post, you must provide: 1. **Label**: 'Stigmatized Content' or 'Neutral Content.' 2. **Explanation**: A brief rationale for why the post was labeled this way. Your decisions must be based on the specific words and tone of the post, avoiding personal interpretation or assumptions about the author's intent unless explicitly stated in the text."},
    {'role': 'user',
               'content': """Here is a social media post. Please classify it as either 'Stigmatized Content' or 'Neutral Content' based on its language and tone regarding people with diabetes. **Post main body**: Tell us the crap you're dealing with this week. Did someone suggest cinnamon again? What about that relative who tried to pray the beetus away?

 As always, please keep in mind [our rules](https://www.reddit.com/r/diabetes/about/rules)"""},
]

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference


inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

["<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are an advanced text analysis AI designed to identify and label social media content for any signs of stigmatized language or messaging regarding people with diabetes. You have expertise in understanding bias, stereotypes, and stigmatizing terms, and you apply your knowledge impartially. When presented with a social media post, you will classify the content based on whether it contains: 1. **Stigmatized Content**: Language that is discriminatory, biased, shaming, or marginalizing towards individuals with diabetes. 2. **Neutral Content**: Language that is neutral, factual, or supportive without stigmatizing. For each post, you must provide: 1. **Label**: 'Stigmatized Content' or 'Neutral Content.' 2. **Explanation**: A brief rationale for why the post was labeled this way. Your decisions must be based on the specific words and tone of the post, avoiding

In [ ]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

**Label:** Neutral Content

**Explanation:** This post appears to be complaining about various frustrating situations related to managing diabetes, but the tone is primarily humorous and observational rather than stigmatizing or discriminatory. The language used, such as "crap" and "what was he thinking?", is colloquial and lighthearted, suggesting that the author is joking about the challenges they face with their diabetes, rather than expressing any hateful or derogatory sentiment towards individuals with diabetes or those living with diabetes-related issues. The reference to "keeping in mind" the rules seems to indicate a nod to existing guidelines or discussions about diabetes management, which
